In [2]:
import os
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

from utils.constants.constants import TEAM_COLORS

## Config

Player-season stats come from basketball-reference's league-wide `totals` page per season
(`/leagues/NBA_{end_year}_totals.html`). That page is not disallowed by basketball-reference's
`robots.txt` (only `*/gamelog/`, `*/splits/`, `*/on-off/`, `*/lineups/`, `*/shooting/` are), and
it already includes everything needed: season totals per team-stint, a combined aggregate row
for traded players (`Tm` = `TOT`/`2TM`/`3TM`), and triple-double counts (`Trp-Dbl`).

Per the site's `robots.txt` (`Crawl-delay: 3`), we sleep 3s between requests. Only 5 requests
total (one per season) are needed, so this whole scrape takes well under a minute.

In [ ]:
DATA_DIR = "data/player_stats"
RAW_DIR = os.path.join(DATA_DIR, "raw")
os.makedirs(RAW_DIR, exist_ok=True)

# 2021-22 through 2025-26 (5 complete seasons), keyed by basketball-reference's season end year
SEASON_END_YEARS = [2022, 2023, 2024, 2025, 2026]

TEAMS = [t for t in TEAM_COLORS.keys() if t != "NBA"]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

CRAWL_DELAY = 3  # seconds, matches basketball-reference robots.txt


def season_label(end_year):
    # "21-22" -- matches the Year column already written to every player_stats CSV
    start = str(end_year - 1)[-2:]
    end = str(end_year)[-2:]
    return f"{start}-{end}"

def nba_api_season_param(end_year):
    # "2021-22" -- the 4-digit-start format nba_api's season= kwarg requires.
    # Only ever pass this into nba_api calls -- never write it to a CSV/DB column.
    return f"{end_year - 1}-{str(end_year)[-2:]}"

In [4]:
def fetch_totals_html(end_year, retries=3, retry_delay=5):
    save_path = os.path.join(RAW_DIR, f"NBA_{end_year}_totals.html")
    if os.path.exists(save_path):
        return save_path

    url = f"https://www.basketball-reference.com/leagues/NBA_{end_year}_totals.html"

    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=30)
            if resp.status_code == 200:
                # write raw bytes; basketball-reference serves UTF-8 but doesn't always
                # declare it, so trusting requests' guessed resp.encoding can mojibake names
                with open(save_path, "wb") as f:
                    f.write(resp.content)
                return save_path
            print(f"HTTP {resp.status_code} for {url} (attempt {attempt}/{retries})")
        except requests.RequestException as e:
            print(f"Error fetching {url}: {e} (attempt {attempt}/{retries})")

        if attempt < retries:
            time.sleep(retry_delay * attempt)

    raise RuntimeError(f"Failed to fetch {url} after {retries} attempts")

In [5]:
for end_year in SEASON_END_YEARS:
    save_path = os.path.join(RAW_DIR, f"NBA_{end_year}_totals.html")
    already_cached = os.path.exists(save_path)

    path = fetch_totals_html(end_year)
    print(f"{season_label(end_year)}: {path}")

    if not already_cached:
        time.sleep(CRAWL_DELAY)

21-22: data/player_stats/raw/NBA_2022_totals.html
22-23: data/player_stats/raw/NBA_2023_totals.html
23-24: data/player_stats/raw/NBA_2024_totals.html
24-25: data/player_stats/raw/NBA_2025_totals.html
25-26: data/player_stats/raw/NBA_2026_totals.html


## Parsing

For a traded player, basketball-reference lists a combined aggregate row (`Tm` = `TOT`/`2TM`/`3TM`/`4TM`)
followed by one row per team stint in chronological order. We use the aggregate row's stats
(full-season combined totals per the user's request) and build the `Team` field by joining the
individual stint team codes in order, e.g. `DAL-LAL`. The final team (last stint) determines which
of the 30 output CSVs the player lands in.

In [6]:
COUNTING_FIELDS = [
    "age", "games", "games_started", "mp", "fg", "fga", "fg3", "fg3a",
    "ft", "fta", "orb", "drb", "trb", "ast", "stl", "blk", "tov", "pf", "pts", "tpl_dbl",
]
PCT_FIELDS = ["fg_pct", "fg3_pct", "ft_pct"]


def to_count(val):
    if val in (None, ""):
        return 0.0
    return float(val)


def to_pct(val):
    if val in (None, ""):
        return None
    return float(val)


def parse_totals_html(path, end_year):
    with open(path, encoding="utf-8") as f:
        html = f.read()

    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table", {"id": "totals_stats"})
    rows = table.find("tbody").find_all("tr")

    # group rows by player_id, preserving the order basketball-reference lists them in
    # (aggregate row first, then each team stint in chronological order)
    player_order = []
    player_rows = {}

    for row in rows:
        if row.get("class") and "thead" in row.get("class"):
            continue

        name_cell = row.find("td", {"data-stat": "name_display"})
        link = name_cell.find("a") if name_cell else None
        if not link:
            continue

        player_id = os.path.basename(link["href"]).replace(".html", "")
        data = {c.get("data-stat"): c.get_text(strip=True) for c in row.find_all(["th", "td"])}
        data["player_id"] = player_id

        if player_id not in player_rows:
            player_rows[player_id] = []
            player_order.append(player_id)
        player_rows[player_id].append(data)

    records = []
    for player_id in player_order:
        stint_rows = player_rows[player_id]

        agg_row = None
        real_stints = []
        for r in stint_rows:
            team = r.get("team_name_abbr", "")
            if team == "TOT" or re.match(r"^\d+TM$", team):
                agg_row = r
            else:
                real_stints.append(r)

        if not real_stints and agg_row is None:
            continue

        if real_stints:
            team_chain = "-".join(r["team_name_abbr"] for r in real_stints)
            final_team = real_stints[-1]["team_name_abbr"]
        else:
            team_chain = agg_row["team_name_abbr"]
            final_team = agg_row["team_name_abbr"]

        stat_row = agg_row if agg_row is not None else real_stints[0]

        record = {
            "player_id": player_id,
            "player_name": stat_row.get("name_display", ""),
            "year_label": season_label(end_year),
            "team": team_chain,
            "final_team": final_team,
            "position": stat_row.get("pos", ""),
        }
        for field in COUNTING_FIELDS:
            record[field] = to_count(stat_row.get(field))
        for field in PCT_FIELDS:
            record[field] = to_pct(stat_row.get(field))

        records.append(record)

    return pd.DataFrame(records)

In [7]:
season_dfs = []
for end_year in SEASON_END_YEARS:
    path = os.path.join(RAW_DIR, f"NBA_{end_year}_totals.html")
    df = parse_totals_html(path, end_year)
    print(f"{season_label(end_year)}: {len(df)} player-seasons")
    season_dfs.append(df)

all_players = pd.concat(season_dfs, ignore_index=True)
all_players.head()

21-22: 605 player-seasons
22-23: 539 player-seasons
23-24: 572 player-seasons
24-25: 569 player-seasons
25-26: 582 player-seasons


,player_id,player_name,year_label,team,final_team,position,age,games,games_started,mp,...,ast,stl,blk,tov,pf,pts,tpl_dbl,fg_pct,fg3_pct,ft_pct
0,youngtr01,Trae Young,21-22,ATL,ATL,PG,23.0,76.0,76.0,2652.0,...,737.0,72.0,7.0,303.0,128.0,2155.0,0.0,0.460,0.382,0.904
1,derozde01,DeMar DeRozan,21-22,CHI,CHI,PF,32.0,76.0,76.0,2743.0,...,374.0,68.0,24.0,181.0,178.0,2118.0,0.0,0.504,0.352,0.877
2,embiijo01,Joel Embiid,21-22,PHI,PHI,C,27.0,68.0,68.0,2297.0,...,284.0,77.0,99.0,214.0,181.0,2079.0,2.0,0.499,0.371,0.814
3,tatumja01,Jayson Tatum,21-22,BOS,BOS,SF,23.0,76.0,76.0,2731.0,...,334.0,75.0,49.0,217.0,174.0,2046.0,0.0,0.453,0.353,0.853
4,jokicni01,Nikola Jokić,21-22,DEN,DEN,C,26.0,74.0,74.0,2476.0,...,584.0,109.0,63.0,281.0,191.0,2004.0,19.0,0.583,0.337,0.810


## Build final output columns and split into 30 team CSVs

In [8]:
def safe_div(a, b):
    return a / b if b else None


def build_output(df):
    games = df["games"]

    out = pd.DataFrame({
        "Player": df["player_name"],
        "Player_ID": df["player_id"],
        "Year": df["year_label"],
        "Team": df["final_team_chain"],
        "Position": df["position"],
        "PPG": df["pts"] / games.replace(0, pd.NA),
        "ORPG": df["orb"] / games.replace(0, pd.NA),
        "RPG": df["trb"] / games.replace(0, pd.NA),
        "APG": df["ast"] / games.replace(0, pd.NA),
        "SPG": df["stl"] / games.replace(0, pd.NA),
        "BPG": df["blk"] / games.replace(0, pd.NA),
        "TOPG": df["tov"] / games.replace(0, pd.NA),
        "FG%": df["fg_pct"],
        "3P%": df["fg3_pct"],
        "FT%": df["ft_pct"],
        "MPG": df["mp"] / games.replace(0, pd.NA),
        "FPG": df["pf"] / games.replace(0, pd.NA),
        "PTS": df["pts"],
        "REB": df["trb"],
        "AST": df["ast"],
        "STL": df["stl"],
        "BLK": df["blk"],
        "TO": df["tov"],
        "FGM": df["fg"],
        "FGA": df["fga"],
        "3PM": df["fg3"],
        "3PA": df["fg3a"],
        "FTM": df["ft"],
        "FTA": df["fta"],
        "MINS": df["mp"],
        "FLS": df["pf"],
        "G": df["games"],
        "GS": df["games_started"],
        "TD": df["tpl_dbl"],
    })

    round_cols = ["PPG", "ORPG", "RPG", "APG", "SPG", "BPG", "TOPG", "MPG", "FPG"]
    out[round_cols] = out[round_cols].round(1)

    # season totals aren't averaged, so they should be whole numbers, not floats
    int_cols = [
        "PTS", "REB", "AST", "STL", "BLK", "TO", "FGM", "FGA", "3PM", "3PA",
        "FTM", "FTA", "MINS", "FLS", "G", "GS", "TD",
    ]
    out[int_cols] = out[int_cols].round().astype(int)

    return out


all_players["final_team_chain"] = all_players["team"]
output_df = build_output(all_players)
output_df.head()

,Player,Player_ID,Year,Team,Position,PPG,ORPG,RPG,APG,SPG,...,FGA,3PM,3PA,FTM,FTA,MINS,FLS,G,GS,TD
0,Trae Young,youngtr01,21-22,ATL,PG,28.4,0.7,3.7,9.7,0.9,...,1544,233,610,500,553,2652,128,76,76,0
1,DeMar DeRozan,derozde01,21-22,CHI,PF,27.9,0.7,5.2,4.9,0.9,...,1535,50,142,520,593,2743,178,76,76,0
2,Joel Embiid,embiijo01,21-22,PHI,C,30.6,2.1,11.7,4.2,1.1,...,1334,93,251,654,803,2297,181,68,68,2
3,Jayson Tatum,tatumja01,21-22,BOS,SF,26.9,1.1,8.0,4.4,1.0,...,1564,230,651,400,469,2731,174,76,76,0
4,Nikola Jokić,jokicni01,21-22,DEN,C,27.1,2.8,13.8,7.9,1.5,...,1311,97,288,379,468,2476,191,74,74,19


In [9]:
merged = output_df.copy()
merged["final_team"] = all_players["final_team"].values

for team in TEAMS:
    team_df = merged[merged["final_team"] == team].drop(columns=["final_team"])
    team_df = team_df.sort_values(["Year", "Player"])
    save_path = os.path.join(DATA_DIR, f"{team}.csv")
    team_df.to_csv(save_path, index=False)
    print(f"{team}: {len(team_df)} player-seasons -> {save_path}")

unassigned = merged[~merged["final_team"].isin(TEAMS)]
if len(unassigned):
    print(f"\nWARNING: {len(unassigned)} rows had an unrecognized team code:")
    print(unassigned[["Player", "Year", "Team"]])

ATL: 96 player-seasons -> data/player_stats/ATL.csv
BOS: 91 player-seasons -> data/player_stats/BOS.csv
BRK: 94 player-seasons -> data/player_stats/BRK.csv
CHO: 97 player-seasons -> data/player_stats/CHO.csv
CHI: 96 player-seasons -> data/player_stats/CHI.csv
CLE: 98 player-seasons -> data/player_stats/CLE.csv
DAL: 102 player-seasons -> data/player_stats/DAL.csv
DEN: 90 player-seasons -> data/player_stats/DEN.csv
DET: 106 player-seasons -> data/player_stats/DET.csv
GSW: 88 player-seasons -> data/player_stats/GSW.csv
HOU: 81 player-seasons -> data/player_stats/HOU.csv
IND: 97 player-seasons -> data/player_stats/IND.csv
LAC: 90 player-seasons -> data/player_stats/LAC.csv
LAL: 94 player-seasons -> data/player_stats/LAL.csv
MEM: 109 player-seasons -> data/player_stats/MEM.csv
MIA: 94 player-seasons -> data/player_stats/MIA.csv
MIL: 95 player-seasons -> data/player_stats/MIL.csv
MIN: 90 player-seasons -> data/player_stats/MIN.csv
NOP: 93 player-seasons -> data/player_stats/NOP.csv
NYK: 92 p